In [10]:
# Load the model
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from dotenv import load_dotenv

# Import the recommender classes
from recommender import AsyncOpenAIClient, CosineSimilarityCalculator, EmbeddingRecommender

# Load environment variables
load_dotenv()

# Configuration setup
config = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "OPENAI_API_VERSION": os.getenv("OPENAI_API_VERSION"),
    "OPENAI_API_BASE": os.getenv("OPENAI_API_BASE"),
    "OPENAI_ORGANIZATION_ID": os.getenv("OPENAI_ORGANIZATION_ID"),
    "GENERATOR_MODEL": os.getenv("GENERATOR_MODEL", "gpt-4"),  # Set default if not in env
    "RECOMMENDER_MODEL": os.getenv("RECOMMENDER_MODEL", "gpt-4"),
    "OPENAI_EMBEDDING_MODEL": os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-ada-002")
}

# Initialize components
openai_client = AsyncOpenAIClient(config)
similarity_calculator = CosineSimilarityCalculator()
recModel = EmbeddingRecommender(openai_client, similarity_calculator)

emb_df = pd.read_pickle('embeddings.pkl')
undergrad = [100, 200, 300, 400, 500]

recModel.load_courses(emb_df)

In [12]:
emb_df

,course,title,description,embedding,level
0,AAS 103,First Year Social Science Seminar,This seminar introduces first-year students to...,"[0.008470812812447548, -0.020093858242034912, ...",100
1,AAS 104,First Year Humanities Seminar,This seminar introduces first-year students to...,"[0.015089535154402256, -0.02548396773636341, 0...",100
2,AAS 111,Introduction to Africa and Its Diaspora,Introduces basic concepts and methods involved...,"[-0.014231206849217415, -0.0060482630506157875...",100
3,AAS 115,Elementary Swahili I,This introductory-level course is designed for...,"[-0.004489186219871044, 0.010212578810751438, ...",100
4,AAS 116,Elementary Swahili II: Language and Culture,This introductory-level course is designed for...,"[0.0031663388945162296, 0.003630136139690876, ...",100
...,...,...,...,...,...
13563,WRITING 410,Quantitative Analysis and Writing in the Disci...,"In various disciplinary iterations, this cours...","[0.00869917031377554, 0.018616488203406334, 0....",400
13564,WRITING 420,Minor in Writing Capstone,"In this course, Minor in Writing students prod...","[0.0156615749001503, -0.016281183809041977, -0...",400
13565,WRITING 630,Advanced Writing for Graduate Students,This advanced writing course for graduate stud...,"[0.01660611853003502, 0.006102879531681538, -0...",600
13566,WRITING 631,Dissertation Writing,This course helps doctoral students make subst...,"[-0.0005177850252948701, 0.005452962126582861,...",600


In [13]:
emb_df['description'].str.split().str.len().mean()

np.float64(50.496615135899766)

In [14]:
# Get word counts for each description
word_counts = emb_df['description'].str.split().str.len()

# Get 5th and 95th percentile values
p5 = word_counts.quantile(0.05)
p95 = word_counts.quantile(0.95)

print(f"5th percentile: {p5}")
print(f"95th percentile: {p95}")

5th percentile: 2.0
95th percentile: 147.0


In [ ]:
q1 = "What courses would you recommend to a second semester freshman student that is an aspiring political science major with interests in public policy, history, natural sciences, and computer programming?"
q2 = "I am a math major interested in computer science theory, what are some courses that may interest me?"

In [7]:
print (await recModel.recommend(q1, levels=undergrad))

INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"


1. **POLSCI 140: Intr Compar Politic**
   Rationale: This course provides a foundational understanding of comparative politics, which is essential for an aspiring political science major interested in public policy. It will help the student develop analytical skills to compare political systems and understand global political dynamics.
   Confidence: High

2. **POLSCI 325 (Cross-listed as PUBPOL 201): Systematic Thinking About the Problems of the Day**
   Rationale: This course offers an introduction to public policy design and analysis, aligning perfectly with the student's interest in public policy and political science. It covers key policy issues and equips students with systematic thinking skills necessary for policy evaluation and creation.
   Confidence: High

3. **POLSCI 300: Quantitative Empirical Methods of Political Science**
   Rationale: This course introduces quantitative methods used in political science, which is crucial for the student to understand empirical research 

In [5]:
print (await recModel.recommend(q1, levels=undergrad))

INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"


1. **POLSCI 325 (Cross-listed as PUBPOL 201): Systematic Thinking About the Problems of the Day**
   Rationale: This course is ideal for an aspiring political science major interested in public policy, as it provides a comprehensive introduction to policy design and analysis using systematic thinking. It covers key policy issues relevant to the student's interests, such as environment, health care, and social welfare.
   Confidence: High

2. **POLSCI 381: Political Science Research Design**
   Rationale: This course will equip the student with essential research skills in political science, focusing on identifying problems and designing methodologies, which are crucial for a career in public policy and political analysis.
   Confidence: High

3. **POLSCI 300: Quantitative Empirical Methods of Political Science**
   Rationale: This course offers an introduction to empirical methods used in political science, aligning well with the student's interest in data-driven approaches to politica

In [6]:
print (await recModel.recommend(q2, levels=undergrad))

INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"


1. **CSE 574 (Cross-listed as EECS 574): Computational Complexity**
Rationale: This course delves into the fundamentals of computation and complexity theory, which are central to computer science theory, making it an excellent fit for a math major interested in exploring theoretical aspects of computer science. It covers topics such as NP-completeness and randomized computation, aligning well with your interest in computer science theory.
Confidence: High

2. **EECS 376: Foundations of Computer Science**
Rationale: This course provides a comprehensive introduction to the theory of computation, including models like finite state machines and Turing machines, which are crucial for understanding computer science theory. It emphasizes computational complexity and NP-hardness, offering a strong theoretical foundation for a math major interested in computer science.
Confidence: High

3. **EECS 477: Introduction to Algorithms**
Rationale: This course focuses on designing efficient algorithms 

In [8]:
print (await recModel.recommend(q2, levels=undergrad))

INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.umgpt.umich.edu/azure-openai-api-unlimited/openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"


1. **CSE 572 (Cross-listed as EECS 572): Randomness and Computation**
Rationale: This course delves into the use of randomness in computer science, covering topics such as algorithms, computational complexity, and cryptography, which align well with your interest in computer science theory. It provides a strong theoretical foundation that complements your math major.
Confidence: High

2. **CSE 574 (Cross-listed as EECS 574): Computational Complexity**
Rationale: As a math major interested in computer science theory, this course offers an in-depth exploration of computational complexity, including NP-completeness and randomized computation, which are crucial for understanding the limits and capabilities of algorithms.
Confidence: High

3. **EECS 376: Foundations of Computer Science**
Rationale: This course introduces the theory of computation, including models like finite state machines and Turing machines, which are essential for understanding the theoretical underpinnings of computer 